# 02 · Datasets — SFT · DPO · Reward · RLDS · WM

Every export is JSONL, so training is just another filter on the pipe. We generate a synthetic
population and export every format.

In [1]:
import json, tempfile, os
from episodic import exporters
from episodic.testing import make_episode, make_population

# A population with repeated intents so DPO can form chosen/rejected pairs.
pop = []
for i in range(6):
    pop.append(make_episode(f"ep_good_{i}", intent="add retry to http client",
                            outcome="merged", feedback=["useful"], passed=3, failed=0))
    pop.append(make_episode(f"ep_bad_{i}", intent="add retry to http client",
                            outcome="reverted", feedback=["wrong"], passed=1, failed=2))
pop += make_population(10, seed=1)
print("episodes:", len(pop), "| formats:", exporters.FORMATS)

episodes: 22 | formats: ('sft', 'dpo', 'reward', 'rlds', 'wm', 'jsonl', 'parquet')


In [2]:
out = tempfile.mkdtemp()
counts = {}
for fmt in exporters.FORMATS:
    res = exporters.export(pop, fmt, os.path.join(out, fmt))
    counts[fmt] = res.get("count")
import pandas as pd
pd.DataFrame(sorted(counts.items()), columns=["format", "rows"])

,format,rows
0,dpo,1
1,jsonl,22
2,parquet,22
3,reward,22
4,rlds,22
5,sft,12
6,wm,49


**SFT** = `intent -> good trajectory` (only episodes that passed the quality bar):

In [3]:
sft = [json.loads(l) for l in open(os.path.join(out, "sft", "sft.jsonl"))]
print("rows:", len(sft))
print(json.dumps(sft[0]["messages"], indent=2)[:600])

rows: 12
[
  {
    "role": "user",
    "content": "add retry to http client"
  },
  {
    "role": "assistant",
    "content": "USER: add retry to http client\nACTION user_prompt(add retry to http client)\nOBS: \nACTION Edit({\"file_path\": \"src/http.py\"}) @ /repo\nOBS: applied edit to src/http.py\nACTION Bash({\"command\": \"pytest -q\"}) @ /repo\nOBS: ============================= test session starts =============================\ncollected 3 items\n\n...\n\n============================== 3 passed in 0.1s =============================="
  }
]


**DPO** = `chosen > rejected` preference pairs grouped by intent:

In [4]:
dpo = [json.loads(l) for l in open(os.path.join(out, "dpo", "dpo.jsonl"))]
print("pairs:", len(dpo))
if dpo:
    print("chosen reward:", dpo[0]["meta"]["chosen_reward"], "| rejected:", dpo[0]["meta"]["rejected_reward"])

pairs: 1
chosen reward: 0.875 | rejected: 0.075


**RLDS** = per-step `(observation, action, next_observation, reward, is_terminal, discount)` transitions:

In [5]:
rlds = [json.loads(l) for l in open(os.path.join(out, "rlds", "rlds.jsonl"))]
step = rlds[0]["steps"][1]
print("action:", step["action"]["tool"], "| reward:", step["reward"], "| terminal:", step["is_terminal"], "| discount:", step["discount"])

action: Edit | reward: 0.0 | terminal: False | discount: 1.0


**WM** (world-model) = `history + action -> next observation`, as SFT messages whose assistant turn IS the observation. `trl-sft` can train a coding world model on this directly.

In [6]:
wm = [json.loads(l) for l in open(os.path.join(out, "wm", "wm.jsonl"))]
print("turn-level samples:", len(wm))
print("user (history+action):\n", wm[0]["messages"][1]["content"][:300])
print("\nassistant (target observation):\n", wm[0]["messages"][2]["content"][:150])

turn-level samples: 49
user (history+action):
 INTENT: add retry to http client
ACTION: user_prompt: add retry to http client
OBSERVATION: 
ACTION: Edit({"file_path": "src/http.py"}) @ /repo
OBSERVATION:

assistant (target observation):
 applied edit to src/http.py
